# Hyperparameter Tuning: Metaheuristic Search vs. scikit-learn Built-ins

`notebooks/applications/hyperparameter_tuning.ipynb` showed that PSO/GA/DE/SMA can drive hyperparameter search by treating it as continuous optimization over $(\log_{10} C, \log_{10}\gamma)$, and `metaheuristics.model_selection.MetaheuristicSearchCV` packages that as a scikit-learn-compatible estimator (`fit`/`predict`/`score`, `best_params_`/`best_score_`/`best_estimator_`), a drop-in alternative to `GridSearchCV`/`RandomizedSearchCV`.

This notebook compares it against scikit-learn's two standard hyperparameter search strategies on the same dataset and evaluation budget:

- **`GridSearchCV`** — evaluates every combination on a fixed lattice. Simple and exhaustive over the grid, but the grid resolution is fixed in advance and doesn't adapt to where good hyperparameters actually are.
- **`RandomizedSearchCV`** — samples `n_iter` combinations independently at random from the search space, ignoring what earlier samples revealed about the landscape.

`MetaheuristicSearchCV` instead adapts its search based on the CV scores it has already observed, so the comparison is about how efficiently each method uses a matched evaluation budget, not about budget differences.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.svm import SVC

from metaheuristics.algorithms.particle_swarm import ParticleSwarmOptimization
from metaheuristics.model_selection import MetaheuristicSearchCV

X, y = load_breast_cancer(return_X_y=True)
PARAM_SPACE = {'C': (1e-2, 1e4, 'log'), 'gamma': (1e-6, 1e1, 'log')}

baseline_accuracy = cross_val_score(SVC(), X, y, cv=5).mean()
print(f'Samples: {X.shape[0]}, features: {X.shape[1]}')
print(f'CV accuracy, default SVC hyperparameters: {baseline_accuracy:.4f}')

Samples: 569, features: 30
CV accuracy, default SVC hyperparameters: 0.9122


## Run the metaheuristic search first, to set the evaluation budget

PSO with the same light budget used in `hyperparameter_tuning.ipynb`-style comparisons (15 particles x 15 iterations). Its evaluation count becomes the budget every scikit-learn method below is matched to.

In [3]:
meta_search = MetaheuristicSearchCV(
    estimator=SVC(),
    param_space=PARAM_SPACE,
    optimizer=ParticleSwarmOptimization(num_particles=15, max_iterations=15),
    cv=5,
    random_state=0,
)
meta_search.fit(X, y)

BUDGET = 15 * (15 + 1)  # (iterations + 1) * particles, matching every method's evaluation count below
print(f'MetaheuristicSearchCV (PSO): {BUDGET} evaluations -> budget for all methods below')
print(f'best CV accuracy={meta_search.best_score_:.4f}, params={meta_search.best_params_}')

MetaheuristicSearchCV (PSO): 240 evaluations -> budget for all methods below
best CV accuracy=0.9561, params={'C': 6950.339629024566, 'gamma': 1e-06}


In [4]:
from scipy.stats import loguniform

grid_resolution = round(np.sqrt(BUDGET))  # lattice sized so grid_resolution ** 2 ~= BUDGET evaluations
grid = GridSearchCV(
    SVC(),
    param_grid={'C': np.logspace(-2, 4, grid_resolution), 'gamma': np.logspace(-6, 1, grid_resolution)},
    cv=5,
).fit(X, y)

random_search = RandomizedSearchCV(
    SVC(),
    param_distributions={'C': loguniform(1e-2, 1e4), 'gamma': loguniform(1e-6, 1e1)},
    n_iter=BUDGET,
    cv=5,
    random_state=0,
).fit(X, y)

results_df = pd.DataFrame([
    {'method': 'MetaheuristicSearchCV (PSO)', 'cv_accuracy': meta_search.best_score_, **meta_search.best_params_},
    {'method': 'GridSearchCV', 'cv_accuracy': grid.best_score_, **grid.best_params_},
    {'method': 'RandomizedSearchCV', 'cv_accuracy': random_search.best_score_, **random_search.best_params_},
]).set_index('method')
results_df

,cv_accuracy,C,gamma
method,,,
MetaheuristicSearchCV (PSO),0.956094,6950.339629,0.000001
GridSearchCV,0.957833,3727.593720,0.000010
RandomizedSearchCV,0.957817,420.227521,0.000011


In [5]:
fig = go.Figure(go.Bar(
    x=['baseline (default hyperparameters)'] + list(results_df.index),
    y=[baseline_accuracy] + list(results_df['cv_accuracy']),
))
fig.update_layout(
    title=f'CV accuracy: baseline vs. each method\'s best hyperparameters ({BUDGET}-evaluation budget)',
    yaxis_title='CV accuracy',
    yaxis_range=[min(results_df['cv_accuracy'].min(), baseline_accuracy) - 0.01, 1.0],
)
fig.show()

## Anytime performance: best-so-far accuracy vs. number of evaluations

Final accuracy can hide how quickly each method got there. `GridSearchCV`/`RandomizedSearchCV` evaluate points independently, so their `cv_results_['mean_test_score']` in evaluation order gives a running best; PSO's `fitness_history` already tracks the swarm's best-so-far after each batch of particle evaluations.

In [6]:
pso_evals = np.arange(1, len(meta_search.result_.fitness_history) + 1) * 15
pso_running_best = 1 - np.array(meta_search.result_.fitness_history)

def running_best(cv_results):
    scores = np.array(cv_results['mean_test_score'])
    return np.arange(1, len(scores) + 1), np.maximum.accumulate(scores)

grid_evals, grid_running_best = running_best(grid.cv_results_)
random_evals, random_running_best = running_best(random_search.cv_results_)

fig = go.Figure()
fig.add_trace(go.Scatter(x=pso_evals, y=pso_running_best, name='MetaheuristicSearchCV (PSO)', mode='lines'))
fig.add_trace(go.Scatter(x=grid_evals, y=grid_running_best, name='GridSearchCV', mode='lines'))
fig.add_trace(go.Scatter(x=random_evals, y=random_running_best, name='RandomizedSearchCV', mode='lines'))
fig.update_layout(
    title='Anytime performance: best-so-far CV accuracy vs. number of evaluations',
    xaxis_title='evaluations',
    yaxis_title='best-so-far CV accuracy',
)
fig.show()

## Takeaways

`GridSearchCV`'s lattice wastes evaluations on combinations far from the optimum, and `RandomizedSearchCV`'s samples never get sharper as evidence accumulates. `MetaheuristicSearchCV` uses each evaluation to steer the next batch of candidates toward promising regions, so it tends to reach a good CV accuracy faster per-evaluation, at the cost of an optimizer with its own hyperparameters (population size, iteration count) to set.